In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ipywidgets import interact, IntSlider

In [2]:
df = pd.read_csv("ring2_remove_baseline_outlier.csv")


In [3]:
# import pandas as pd
# df = pd.read_csv("../../5_Data/ring2.csv")
# df.to_parquet("../../5_Data/ring2.parquet")

In [4]:
import os
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

In [5]:
os.listdir()

['.ipynb_checkpoints',
 'extract_peaks.py',
 'features.py',
 'highpass.py',
 'llxaa.ipynb',
 'output.parquet',
 'peaks.parquet',
 'ring2_remove_baseline_outlier.csv',
 'rri.parquet']

In [6]:
re = pd.read_parquet("output.parquet")

In [7]:
re

,date,time,duration,red,ied,accX,accY,accZ,ied_despiked,ied_clean
0,01-04-2026,22:47:00,0:00:00,16186800,16234168,812,47,1949,16234168.0,-172939.357439
1,01-04-2026,22:47:00,0:00:00,16187639,16238248,812,47,1949,16238248.0,-171034.499170
2,01-04-2026,22:47:00,0:00:00,16188541,16240329,812,47,1949,16240329.0,-168931.729276
3,01-04-2026,22:47:00,0:00:00,16189901,16241725,812,47,1949,16241725.0,-166634.209922
4,01-04-2026,22:47:00,0:00:00,16190749,16243498,812,47,1949,16243498.0,-164146.411032
...,...,...,...,...,...,...,...,...,...,...
1968295,02-04-2026,04:16:12,5:29:12,13508395,13651079,-1621,-159,1562,13651079.0,6355.385361
1968296,02-04-2026,04:16:12,5:29:12,13506294,13648198,-1616,-150,1554,13648198.0,5114.146043
1968297,02-04-2026,04:16:12,5:29:12,13504781,13645853,-1631,-153,1545,13645853.0,3595.190950
1968298,02-04-2026,04:16:12,5:29:12,13503444,13643924,-1618,-133,1552,13643924.0,1898.473527


In [8]:
pp = pd.read_parquet("peaks.parquet")

In [9]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider

# 假设你的完整结果读入为 pp
# pp = pd.read_parquet("peaks.parquet")

# 提取信号和波峰、波谷布尔掩码
y = -1. * pp['ied_clean'].values  # 注意：这里信号取了反，视觉上的高低点会与原信号相反
is_peak_array = pp['is_peak'].values  
is_valley_array = pp['is_valley'].values  # 新增：提取波谷掩码

@interact(
    start=IntSlider(min=0, max=len(y)-200, step=30, value=0, description='start idx'),
    window=IntSlider(min=200, max=100000, step=100, value=500, description='window')
)
def plot_zoom(start=0, window=500):
    end = min(start + window, len(y))
    
    # 构建当前窗口的时间轴 (s)
    t = np.arange(start, end) / 100.0
    
    # 1. 截取当前窗口的信号
    y_slice = y[start:end]
    
    # 2. 截取波峰，找出对应时间和坐标
    peak_mask_slice = is_peak_array[start:end]
    peak_indices = np.where(peak_mask_slice)[0] 
    t_peaks = t[peak_indices]
    y_peaks = y_slice[peak_indices]

    # 3. 截取波谷，找出对应时间和坐标 (新增)
    valley_mask_slice = is_valley_array[start:end]
    valley_indices = np.where(valley_mask_slice)[0]
    t_valleys = t[valley_indices]
    y_valleys = y_slice[valley_indices]

    plt.rcParams['font.size'] = 18
    fig, ax = plt.subplots(figsize=(18, 7))
    
    # 画基础信号连线
    ax.plot(t, y_slice, 'o-', linewidth=1.5, color='tab:blue', label='IED (Inverted)', markersize=4)
    
    # 画波峰标记 (用红色的大叉号标记)
    ax.plot(t_peaks, y_peaks, 'rx', markersize=12, markeredgewidth=2.5, label='Detected Peaks')

    # 画波谷标记 (用绿色的空心圆圈标记，新增)
    ax.plot(t_valleys, y_valleys, 'go', markersize=10, markeredgewidth=2.5, fillstyle='none', label='Detected Valleys')

    ax.set_xlabel('Time (s)')
    ax.set_ylabel('IED (high-pass inverted)')
    ax.set_title(f'Sample {start}~{end} ({window} samples / {window/100:.1f}s)')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='upper right')
    
    plt.tight_layout()
    plt.show()

interactive(children=(IntSlider(value=0, description='start idx', max=1968100, step=30), IntSlider(value=500, …

In [10]:
pp.head()

,date,time,duration,red,ied,accX,accY,accZ,ied_despiked,ied_clean,is_peak,peak_value,is_valley,valley_value,rr_interval_ms,vv_interval_ms,branch_phase
0,01-04-2026,22:47:00,0:00:00,16186800,16234168,812,47,1949,16234168.0,-172939.357439,False,NaN,False,NaN,NaN,NaN,unknown
1,01-04-2026,22:47:00,0:00:00,16187639,16238248,812,47,1949,16238248.0,-171034.499170,False,NaN,False,NaN,NaN,NaN,unknown
2,01-04-2026,22:47:00,0:00:00,16188541,16240329,812,47,1949,16240329.0,-168931.729276,False,NaN,False,NaN,NaN,NaN,unknown
3,01-04-2026,22:47:00,0:00:00,16189901,16241725,812,47,1949,16241725.0,-166634.209922,False,NaN,False,NaN,NaN,NaN,unknown
4,01-04-2026,22:47:00,0:00:00,16190749,16243498,812,47,1949,16243498.0,-164146.411032,False,NaN,False,NaN,NaN,NaN,unknown


In [11]:
pp

,date,time,duration,red,ied,accX,accY,accZ,ied_despiked,ied_clean,is_peak,peak_value,is_valley,valley_value,rr_interval_ms,vv_interval_ms,branch_phase
0,01-04-2026,22:47:00,0:00:00,16186800,16234168,812,47,1949,16234168.0,-172939.357439,False,NaN,False,NaN,NaN,NaN,unknown
1,01-04-2026,22:47:00,0:00:00,16187639,16238248,812,47,1949,16238248.0,-171034.499170,False,NaN,False,NaN,NaN,NaN,unknown
2,01-04-2026,22:47:00,0:00:00,16188541,16240329,812,47,1949,16240329.0,-168931.729276,False,NaN,False,NaN,NaN,NaN,unknown
3,01-04-2026,22:47:00,0:00:00,16189901,16241725,812,47,1949,16241725.0,-166634.209922,False,NaN,False,NaN,NaN,NaN,unknown
4,01-04-2026,22:47:00,0:00:00,16190749,16243498,812,47,1949,16243498.0,-164146.411032,False,NaN,False,NaN,NaN,NaN,unknown
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1968295,02-04-2026,04:16:12,5:29:12,13508395,13651079,-1621,-159,1562,13651079.0,6355.385361,False,NaN,False,NaN,NaN,NaN,unknown
1968296,02-04-2026,04:16:12,5:29:12,13506294,13648198,-1616,-150,1554,13648198.0,5114.146043,False,NaN,False,NaN,NaN,NaN,unknown
1968297,02-04-2026,04:16:12,5:29:12,13504781,13645853,-1631,-153,1545,13645853.0,3595.190950,False,NaN,False,NaN,NaN,NaN,unknown
1968298,02-04-2026,04:16:12,5:29:12,13503444,13643924,-1618,-133,1552,13643924.0,1898.473527,False,NaN,False,NaN,NaN,NaN,unknown


In [12]:
pd.unique(pp.branch_phase)

<ArrowStringArray>
['unknown', 'descending', 'ascending']
Length: 3, dtype: str

In [13]:
fea = pd.read_parquet("features.parquet")

In [15]:
fea.head(20)

,timestamp,rri_ms,asc_area,desc_area,motion
0,01-04-2026 22:47:01,1100.0,1.337497e+07,3.952694e+07,2097.386514
1,01-04-2026 22:47:02,820.0,1.634264e+07,4.552597e+06,2012.349159
2,01-04-2026 22:47:05,810.0,4.198418e+05,1.570207e+06,1907.738494
3,01-04-2026 22:47:06,780.0,4.591512e+05,1.389924e+06,2200.066015
4,01-04-2026 22:47:06,740.0,2.267045e+05,8.669536e+05,2261.837840
5,01-04-2026 22:47:07,710.0,2.781777e+05,8.090771e+05,2257.098751
6,01-04-2026 22:47:08,740.0,2.602608e+05,9.215927e+05,2231.763019
7,01-04-2026 22:47:09,790.0,2.769366e+05,1.059257e+06,2238.236008
8,01-04-2026 22:47:09,790.0,3.092077e+05,1.144490e+06,2260.750687
9,01-04-2026 22:47:10,800.0,2.664359e+05,1.066004e+06,2235.755126
